In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [38]:
data= pd.read_csv("..\\data\\winemag-data-130k-v2.csv")

data.info()


<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  str    
 2   description            129971 non-null  str    
 3   designation            92506 non-null   str    
 4   points                 129951 non-null  float64
 5   price                  129971 non-null  int64  
 6   province               129908 non-null  str    
 7   region_1               108724 non-null  str    
 8   region_2               50511 non-null   str    
 9   taster_name            103727 non-null  str    
 10  taster_twitter_handle  98758 non-null   str    
 11  title                  129971 non-null  str    
 12  variety                129970 non-null  str    
 13  winery                 129971 non-null  str    
dtypes: float64(1), int64(2), str(11)
memory usage: 

In [39]:
y=data.price
print(y.info())

<class 'pandas.Series'>
RangeIndex: 129971 entries, 0 to 129970
Series name: price
Non-Null Count   Dtype
--------------   -----
129971 non-null  int64
dtypes: int64(1)
memory usage: 1015.5 KB
None


In [40]:


melb_predictors = data.drop(['price'], axis=1)
print(melb_predictors.info())

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  str    
 2   description            129971 non-null  str    
 3   designation            92506 non-null   str    
 4   points                 129951 non-null  float64
 5   province               129908 non-null  str    
 6   region_1               108724 non-null  str    
 7   region_2               50511 non-null   str    
 8   taster_name            103727 non-null  str    
 9   taster_twitter_handle  98758 non-null   str    
 10  title                  129971 non-null  str    
 11  variety                129970 non-null  str    
 12  winery                 129971 non-null  str    
dtypes: float64(1), int64(1), str(11)
memory usage: 60.1 MB
None


In [41]:
X= melb_predictors.select_dtypes(exclude=['object'])

print(X.info())

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Unnamed: 0  129971 non-null  int64  
 1   points      129951 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 2.0 MB
None


In [28]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [13]:
def score_dataset(X_train,X_test,y_train,y_test):
    model=RandomForestRegressor(n_estimators=10,random_state=0)
    model.fit(X_train,y_train)
    preds=model.predict(X_test)
    return mean_absolute_error(y_test,preds)


In [42]:
X.info()
X_train, X_test,y_train,y_test=train_test_split(X,y,train_size=.8,test_size=0.2, random_state=0)

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Unnamed: 0  129971 non-null  int64  
 1   points      129951 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 2.0 MB


In [43]:
# Get names of columns with missing values
cols_with_missing = [col for col in X_train.columns
                     if X_train[col].isnull().any()]
print(cols_with_missing)

['points']


In [33]:
print(y_train.info())

<class 'pandas.Series'>
Index: 103976 entries, 121599 to 68268
Series name: price
Non-Null Count  Dtype  
--------------  -----  
96786 non-null  float64
dtypes: float64(1)
memory usage: 1.6 MB
None


In [44]:

# Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_test.drop(cols_with_missing, axis=1)

print("MAE from Approach 1 (Drop columns with missing values):")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_test))

MAE from Approach 1 (Drop columns with missing values):
18.25718407386036


In [46]:
#Approach 2 : Simple Imputer
from sklearn.impute import SimpleImputer

my_imputer=SimpleImputer()

imputed_X_train=pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_test=pd.DataFrame(my_imputer.fit_transform(X_test))

imputed_X_train.columns=X_train.columns
imputed_X_test.columns=X_test.columns

print("MAE from Approach 2 (Imputation):")
print(score_dataset(imputed_X_train, imputed_X_test, y_train, y_test))

MAE from Approach 2 (Imputation):
18.038649740334677


In [ ]:
# Make copy to avoid changing original data (when imputing)
X_train_plus = X_train.copy()
X_valid_plus = X_test.copy()

# Make new columns indicating what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()

# Imputation
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_valid_plus = pd.DataFrame(my_imputer.transform(X_valid_plus))

# Imputation removed column names; put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

print("MAE from Approach 3 (An Extension to Imputation):")
print(score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_test))